# ÍSKOS 2024 — 00. Forvinnsla

Markmið þessarar vinnubókar er einfalt: lesa SPSS-skrá ÍSKOS 2024, einangra svarendur í **ÍsKos-bálknum** (n ≈ 1193), búa til markmiðsbreytuna `flokkur` og vista hreina CSV-skrá sem næstu vinnubækur lesa.

Við látum hópa 4 (málefnaeign) og 5 (flokkshollusta og leiðtogar) fljóta með í CSV-skránni þó að við sleppum þeim í líkanavinnubókinni — það er gert vísvitandi svo þú getir prófað sjálf(ur) að bæta þeim aftur við og séð hvernig **leakage** kemur fram.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import pyreadstat

# Project paths — notebook lives in notebooks_simplified/, so go up one level.
PROJECT = Path.cwd().parent
DATA_PATH = PROJECT / "Kjósendakönnun" / "ÍSKOS kjósenda 2024" / "ISKOS2024_med_aukabreytum_20260318.sav"
GROUPS_PATH = PROJECT / "iskos24_breytuflokkun.csv"
OUT_DIR = PROJECT / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_PATH exists:", DATA_PATH.exists())
print("GROUPS_PATH exists:", GROUPS_PATH.exists())

## 1. Lesum .sav-skrána

Við notum `pyreadstat.read_sav(..., apply_value_formats=False, user_missing=True)`:

- `apply_value_formats=False` heldur tölulegum kóðum (1, 2, 3, ...) í gögnunum í stað þess að skipta þeim út fyrir textamerkimiða strax. Við gerum þá útskiptingu sjálf síðar, fyrir **flokkabreytur** en ekki fyrir raðbreytur.
- `user_missing=True` heldur SPSS sérstöku missing-kóðunum (995 *Ekki spurt*, 996 *Á ekki við*, 997 *Neitar að svara*, 998 *Veit ekki*, 999 *Brottfall*) inni sem tölum. Ákvörðunartré geta klofið á þeim eins og hverjum öðrum gildum — engin þörf á impútaði.

In [ ]:
df_raw, meta = pyreadstat.read_sav(
    str(DATA_PATH),
    apply_value_formats=False,
    user_missing=True,
)
print(f"Hrá gögn: {df_raw.shape[0]} svarendur, {df_raw.shape[1]} breytur")

## 2. Síum á ÍsKos-bálkinn

Breytan `module` gefur til kynna hvaða spurningalisti var lagður fyrir hvern svaranda — `1 = CSES`, `2 = ÍsKos`, `3 = bæði`. Við viljum eingöngu þá sem fengu ÍsKos-spurningalistann.

In [ ]:
df = df_raw[df_raw["module"] == 2].copy()
print(f"Eftir síun á module==2 (ÍsKos): {len(df)} svarendur")

## 3. Byggjum markmiðið `flokkur`

Markmiðsbreytan er svarið við spurningunni *„Hvern kaust þú?“* (`prtvote24`). Við gerum þrennt:

1. **Sleppum** þeim sem kusu ekki, neituðu að svara, skiluðu auðu, eða kusu örflokk (Lýðræðisflokkinn með aðeins 8 svarendur).
2. **Sameinum** Vinstri-græn (V), Pírata (P) og Sósíalistaflokkinn (J) í einn hóp `„Vinstri (sameinað)“` — annars fáum við flokk með aðeins 30–50 svarendum sem líkanið ræður ekkert við.
3. **Þýðum** tölulegu kóðana í íslensk flokksheiti með því að fletta upp í `meta.variable_value_labels`.

Eftirstandandi 7 flokkar: Samfylkingin, Sjálfstæðisflokkurinn, Viðreisn, Miðflokkurinn, Flokkur fólksins, Framsóknarflokkurinn, og sameinaði vinstrihópurinn.

In [ ]:
# Codes we drop: 0 (kaus ekki), 13 (Lýðræðisflokkur — too small), 990 (annan), 992 (vill ekki segja),
# 993 (auður), 997 (neitar), 998 (veit ekki).
DROP_CODES = {0, 13, 990, 992, 993, 997, 998}
# V, P, J get merged into one left-leaning class.
LEFT_MERGE = {4, 6, 12}

vote_labels = meta.variable_value_labels["prtvote24"]

def to_flokkur(code):
    if pd.isna(code):
        return None
    c = int(code)
    if c in DROP_CODES:
        return None
    if c in LEFT_MERGE:
        return "Vinstri (sameinað)"
    return vote_labels[float(c)]

df["flokkur"] = df["prtvote24"].map(to_flokkur)
df = df.dropna(subset=["flokkur"]).reset_index(drop=True)
print(f"Eftir að við héldum aðeins 7 flokkum: {len(df)} svarendur\n")
print(df["flokkur"].value_counts())

## 4. Veljum breytur úr `iskos24_breytuflokkun.csv`

CSV-skráin merkir hverja breytu með **hópi** (1 – Félagsleg staða, 2 – Málefnaviðhorf, ...) og hvort hún var lögð fyrir í ÍsKos- og/eða CSES-bálkunum. Við höldum öllum breytum sem voru spurðar í ÍsKos (allir hópar 1–8); í líkanavinnubókinni sleppum við svo hópum 4 og 5 vegna leakage.

Skráin er á `cp1252` kóðun (Excel-arfleifð) — þurfum því að segja `pandas` það beint.

In [ ]:
import csv

KEEP_GROUPS = {
    "1 – Félagsleg staða",
    "2 – Málefnaviðhorf",
    "3 – Frammistaða stjórnvalda",
    "4 – Málefnaeign",
    "5 – Flokkshollusta og leiðtogar",
    "6 - Viðhorf og hegðun",
    "7 - Miðlar",
    "8 - Hægri/Vinstri",
}

with open(GROUPS_PATH, encoding="cp1252", newline="") as f:
    grp_rows = list(csv.DictReader(f))

feat_rows = [
    r for r in grp_rows
    if r["iskos"].strip().upper() == "TRUE"
    and r["group2"].strip() in KEEP_GROUPS
]
features = [r["variable"] for r in feat_rows]
print(f"Höldum {len(features)} breytum úr ÍsKos")

## 5. Flokkum dtypes handvirkt

Hverri breytu er úthlutað dtype: **nominal** (flokkabreyta, t.d. kjördæmi, kyn) eða **ordinal** (raðbreyta eða töluleg, t.d. Likert-kvarði, aldur).

- Nominal-breytur fá íslensk **strengjagildi** í CSV-skránni (`„Kona“`, `„Norðvesturkjördæmi“`) — EDA-myndir verða strax læsilegar.
- Ordinal-breytur halda **heiltölugildum** (1, 2, 3, ...) — tré klofnar á þeim í náttúrulegri röð.

Listinn hér að neðan er handvirkur og er sjálfsagður staður til að breyta ef þér finnst eitthvað rangt flokkað.

In [ ]:
NOMINAL = {
    "elecdist", "gender", "rentown", "maritals", "medium",
    "mipmtaskprt", "pidsupport3", "prtcls",
    "bprteco", "bprtimm", "bprteurope", "bprtfamily", "bprtisencit",
    "bprthealth", "bprtenvir", "bprtregpol", "bprtedu", "bprtequal",
    "bprthouse",
    # Employment-type questions: codes are sector/arrangement labels, not magnitudes.
    "govpriv", "uniwindw",
    # Education: parallel tracks (e.g. vocational vs. academic) and "Annað" have no single order.
    "educationflo",
}

def dtype_for(var: str) -> str:
    return "nominal" if var in NOMINAL else "ordinal"

# Quick summary
from collections import Counter
print(Counter(dtype_for(v) for v in features))

## 6. Vinnandi dæmi: endurkóðum `gender`

Áður en við gerum þetta á allar breytur í lykkju skoðum við eina breytu vandlega.

1. Hverjir eru hrákóðarnir?
2. Hvað þýða þeir samkvæmt SPSS-merkimiðum?
3. Hvernig viljum við að gildin líti út í lokaCSV-skránni?

Sumar breytur nota sérstaka kóða utan 995–999 fyrir „ekki svar“ — t.d. `991 = „Þekkir ekki flokkinn“` í `prtfeel_*` eða `97 = „Neitar að svara“` í `bprtequal`. Í stað þess að halda lista af tölum, **þekkjum við þá á íslensku lýsingunni** sem fylgir hverri kóða: ef hún inniheldur *„ekki spurt“*, *„á ekki við“*, *„neitar“*, *„veit ekki“*, *„brottfall“* eða *„þekkir ekki“* — þá er það kóði fyrir vantað svar.

In [ ]:
var = "gender"
print("Hráir kóðar:", sorted(df[var].dropna().unique().tolist()))
print("Gildakóðun:", meta.variable_value_labels[var])

# Detect missing-style codes from the *value label text* rather than hard-coding numbers.
# Different breytur nota mismunandi tölulega kóða (995, 991, 99, 98, 97, ...) en alltaf
# með íslenskri lýsingu sem inniheldur eitt af eftirfarandi orðasamböndum:
MISSING_LABEL_KEYWORDS = (
    "ekki spurt", "á ekki við", "neitar", "veit ekki",
    "brottfall", "þekkir ekki",
)

def missing_codes_for(var: str) -> set[int]:
    """Codes whose value-label text marks them as a non-answer."""
    out = set()
    for code, lbl in meta.variable_value_labels.get(var, {}).items():
        if lbl and any(kw in lbl.lower() for kw in MISSING_LABEL_KEYWORDS):
            out.add(int(code))
    return out

def recode_gender(code):
    if pd.isna(code) or int(code) in missing_codes_for(var):
        return "Ekki svarað"
    return meta.variable_value_labels[var][float(int(code))]

df[var].head(10).map(recode_gender).tolist()

## 7. Sömu rökfræði á allar breytur í lykkju

Hver breyta fer í gegnum sömu meðferð:

- **Nominal:** kóði → íslenskt textaheiti. Missing-kóðar 995–999 fá `„Ekki svarað“`.
- **Ordinal:** kóði → heiltala. Missing-kóðar 995–999 fá `-1` (negative sentinel sem tré geta klofið á).

Niðurstöðu söfnum við í eitt `dict` og smíðum DataFrame undir lokin.

In [ ]:
recoded = {"flokkur": df["flokkur"].values}

for var in features:
    dtype = dtype_for(var)
    vlbl = meta.variable_value_labels.get(var, {})
    missing = missing_codes_for(var)   # per-variable!
    col = df[var]
    if dtype == "nominal":
        def to_str(code, vlbl=vlbl, missing=missing):
            if pd.isna(code) or int(code) in missing:
                return "Ekki svarað"
            return vlbl.get(float(int(code)), str(int(code)))
        recoded[var] = col.map(to_str).astype("string")
    else:  # ordinal / numeric
        def to_int(code, missing=missing):
            if pd.isna(code):
                return -1
            c = int(code)
            return -1 if c in missing else c
        recoded[var] = col.map(to_int).astype("int64")

clean = pd.DataFrame(recoded)
print("Lokastærð:", clean.shape)
clean.head()

## 8. Endurnefnum dálka — íslenskar spurningar í stað stuttra breytuheita

Þegar við skoðum gögn í EDA eða lesum SHAP-mynd er gagnlegra að sjá *„Æskilegt að Ísland gangi í ESB“* en `EUmship`. Þess vegna **skiptum við út** stuttu breytuheitunum fyrir fullu íslensku spurningarnar úr ÍSKOS-spurningalistanum áður en við vistum CSV-skrána. `feature_meta` heldur líka utan um sömu uppfletti, svo næstu vinnubækur geta lykkjað yfir breyturnar með íslenskum lyklum.

`flokkur` (markmiðsbreytan) er ekki spurning sem var lögð fyrir — því heldur sá dálkur sínu nafni.

In [ ]:
# Build short-code → Icelandic-question lookup from breytuflokkun.csv
rename_map = {r["variable"]: r["label"] for r in feat_rows}

# Sanity: no duplicate question texts (would silently merge columns)
from collections import Counter
dupes = [q for q, n in Counter(rename_map.values()).items() if n > 1]
assert not dupes, f"Duplicate question texts found: {dupes}"

clean = clean.rename(columns=rename_map)
clean.head()

## 9. Vistum hreinu CSV-skrána og lýsigögn

Tvær skrár fara í `data/`:

- `iskos_clean.csv` — eitt skjal með öllum breytum (íslenskar spurningar sem dálkanöfn) + `flokkur`.
- `feature_meta.csv` — fyrir hverja breytu: stutt nafn (`short_name`), íslensk spurning (`label` — sem er nú líka dálkanafnið í CSV), hópur, dtype. Næstu vinnubækur lykkja yfir `label` til að ná í dtype/hóp og fá þannig allt sem þær þurfa.

In [ ]:
# Preserve any hand-curated short_label values from a previous run of this notebook.
# Without this, re-running preprocess would wipe the short forms that NB03's SHAP plots rely on.
prev_meta_path = OUT_DIR / "feature_meta.csv"
prev_short = {}
if prev_meta_path.exists():
    try:
        _prev = pd.read_csv(prev_meta_path)
        if "short_label" in _prev.columns:
            prev_short = {
                row["short_name"]: (row["short_label"] if isinstance(row["short_label"], str) else "")
                for _, row in _prev.iterrows()
            }
    except Exception as e:
        print(f"  (could not read previous feature_meta.csv: {e})")

meta_df = pd.DataFrame([
    {
        "short_name": r["variable"],
        "label": r["label"],
        # `short_label` is used by NB03's SHAP plots to keep axis labels readable.
        # Preserved from previous feature_meta.csv when present; default empty otherwise →
        # NB03 falls back to the full Icelandic question text in `label`.
        "short_label": prev_short.get(r["variable"], ""),
        "group": r["group2"].strip(),
        "dtype": dtype_for(r["variable"]),
    }
    for r in feat_rows
])

clean.to_csv(OUT_DIR / "iskos_clean.csv", index=False)
meta_df.to_csv(OUT_DIR / "feature_meta.csv", index=False)
print("Vistað:")
print(" ", OUT_DIR / "iskos_clean.csv")
print(" ", OUT_DIR / "feature_meta.csv")

# --- Value-label sidecar for NB03 figures (Mynd 1, Mynd 4) ---
# Map each ordinal feature's numeric code → human-readable Icelandic answer text (e.g. 4 → "frekar
# óæskilegt"). Keyed by the Icelandic question text so plotting code can look it up by column name.
# Missing-style codes (-1) get mapped to "Ekki svarað" automatically.
value_labels_out = {}
for r in feat_rows:
    var = r["variable"]
    dtype = dtype_for(var)
    if dtype != "ordinal":
        continue
    vlbl = meta.variable_value_labels.get(var, {})
    missing = missing_codes_for(var)
    code_to_text = {}
    for code, text in vlbl.items():
        c = int(code)
        if c in missing:
            continue
        code_to_text[c] = text
    code_to_text[-1] = "Ekki svarað"
    if code_to_text:
        value_labels_out[r["label"]] = code_to_text

with open(OUT_DIR / "value_labels.json", "w", encoding="utf-8") as f:
    json.dump(value_labels_out, f, ensure_ascii=False, indent=2)
print(f"  + value_labels.json ({len(value_labels_out)} ordinal variables)")

## 9. Lokaeftirlit

In [ ]:
print("Flokksskipting:")
print(clean["flokkur"].value_counts())
print("\nDtype yfirlit (frá pandas):")
print(clean.dtypes.value_counts())
print("\nfeature_meta yfirlit:")
print(meta_df["dtype"].value_counts())
print("\nDæmi:")
clean.sample(5, random_state=0)